In [ ]:
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from rich import print as rprint
from loguru import logger
from dotenv import load_dotenv
load_dotenv(override=True)

model = ChatOpenAI(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    final_output: str

class InputState(TypedDict):
    topic: str

class OutputState(TypedDict):
    final_output: str

def node_poem(state: InputState) -> OverAllState:
    logger.info(f"node_poem 已执行")
    topic = state["topic"]
    poem = model.invoke([HumanMessage(f"写一首关于 {topic} 的七言绝句")]).content

    return {
        "poem": poem
    }

def node_joke(state: InputState) -> OverAllState:
    logger.info(f"node_joke 已执行")
    topic = state["topic"]
    joke = model.invoke([HumanMessage(f"写一个关于 {topic} 的笑话")]).content

    return {
        "joke": joke
    }

def node_output(state: OverAllState) -> OutputState:
    logger.info("node_output 已执行")
    topic = state["topic"]
    poem = state["poem"]
    joke = state["joke"]
    final_output = f"关于 {topic} 的七言绝句：\n{poem}\n笑话：\n{joke}"

    return {
        "final_output": final_output
    }

builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)
builder.add_node("node_poem", node_poem)
builder.add_node("node_joke", node_joke)
builder.add_node("node_output", node_output)
builder.add_edge(START, "node_poem")
builder.add_edge(START, "node_joke")
builder.add_edge("node_poem", "node_output")
builder.add_edge("node_joke", "node_output")
builder.add_edge("node_output", END)

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "123"}}
graph = builder.compile(checkpointer=checkpointer)
res = graph.invoke({"topic": "莲花"}, config=config)

from IPython.display import display
display(graph)

print('=' * 30, '-> 运行结果 <-', '=' * 30)
rprint("res: {}", res)

print('=' * 30, '-> 历史检查点列表 <-', '=' * 30)
history_checkpoints = list(graph.get_state_history(config=config))
rprint(history_checkpoints)

In [5]:
last_checkpoints = list(graph.get_state(config=config))
rprint(last_checkpoints)

[
    {
        'topic': '莲花',
        'poem': 
'《莲花》\n玉立亭亭水中央，素影涵波暗自香。\n不共春风争颜色，独向秋潭理素妆。\n\n赏析：这首作品以莲花为吟咏对象，通
过“玉立亭亭”、“素影涵波”等意象勾勒出莲花的清雅姿态。后两句“不共春风争颜色，独向秋潭理素妆”，巧妙运用对比手法，将莲
花不与百花争春的淡泊品格与独自在秋潭中保持本真的高洁气质相映衬，展现了莲花超然物外的精神境界。',
        'joke': 
'莲花最近很不开心，因为大家老拿她和别人比。\n\n有一天，荷叶安慰她：“别郁闷了，你出淤泥而不染，多高尚啊！”\n\n莲花叹
了口气：“高尚有什么用？隔壁那个菊花，人家一夸就是‘采菊东篱下’，多雅致；楼上那个梅花，更是‘凌寒独自开’，多励志。”\n\
n荷叶说：“你也有啊，‘接天莲叶无穷碧，映日荷花别样红’！”\n\n莲花翻了个白眼：“那夸的都是你荷叶好吗？‘接天莲叶’是谁？‘
无穷碧’是谁？我顶多算个‘别样红’，还是个配角！”\n\n荷叶刚想再劝，莲花又说：“最气人的是牡丹，人家直接叫‘花中之王’，我
呢？花中‘廉’洁代表——每次说到我，都跟反腐倡廉绑一块儿，搞得我跟纪委似的。”\n\n这时，池塘边一个小男孩跑过来，对着莲花
大喊：“妈妈快看！那个荷——花——蛋——！”\n\n莲花当场崩溃：“是莲！花！……算了，‘荷花蛋’就‘荷花蛋’吧，至少不是‘反腐蛋’。”'
,
        'final_output': '关于 莲花 
的七言绝句：\n《莲花》\n玉立亭亭水中央，素影涵波暗自香。\n不共春风争颜色，独向秋潭理素妆。\n\n赏析：这首作品以莲花
为吟咏对象，通过“玉立亭亭”、“素影涵波”等意象勾勒出莲花的清雅姿态。后两句“不共春风争颜色，独向秋潭理素妆”，巧妙运用
对比手法，将莲花不与百花争春的淡泊品格与独自在秋潭中保持本真的高洁气质相映衬，展现了莲花超然物外的精神境界。\n笑话
：\n莲花最近很不开心，因为大家老拿她和别人比。\n\n有一天，荷叶安慰她：“别郁闷了，你出淤泥而不染，多高尚啊！”\n\n莲
花叹了口气：“高尚有什么用？隔壁那个菊花，人家一夸就是‘采菊东篱下’，多雅致；楼上那个梅花，更是‘凌寒独自开’，多励志。
”\n\n荷叶说：“你也有啊，‘接天莲叶无穷碧，映日荷花别样红’！”\n\n莲花翻了个白眼：“那夸的都是你荷叶好吗？‘接天莲叶’是
谁？‘无穷碧’是谁？我顶多算个‘别样红’，还是个配角！”\n\n荷叶刚想再劝，莲花又说：“最气人的是牡丹，人家直接叫‘花中之王
’，我呢？花中‘廉’洁代表——每次说到我，都跟反腐倡廉绑一块儿，搞得我跟纪委似的。”\n\n这时，池塘边一个小男孩跑过来，对
着莲花大喊：“妈妈快看！那个荷——花——蛋——！”\n\n莲花当场崩溃：“是莲！花！……算了，‘荷花蛋’就‘荷花蛋’吧，至少不是‘反腐
蛋’。”'
    },
    (),
    {
        'configurable': {
            'thread_id': '123',
            'checkpoint_ns': '',
            'checkpoint_id': '1f1ad0f4-26e7-6ae8-8002-66df0221fa71'
        }
    },
    {'source': 'loop', 'step': 2, 'parents': {}},
    '2026-09-10T12:00:47.584718+00:00',
    {
        'configurable': {
            'thread_id': '123',
            'checkpoint_ns': '',
            'checkpoint_id': '1f1ad0f4-26e2-6098-8001-e34e38d800a5'
        }
    },
    (),
    ()
]

In [6]:
history_checkpoints = list(graph.get_state_history(config=config))
rprint(history_checkpoints)

[
    StateSnapshot(
        values={
            'topic': '莲花',
            'poem': 
'《莲花》\n玉立亭亭水中央，素影涵波暗自香。\n不共春风争颜色，独向秋潭理素妆。\n\n赏析：这首作品以莲花为吟咏对象，通
过“玉立亭亭”、“素影涵波”等意象勾勒出莲花的清雅姿态。后两句“不共春风争颜色，独向秋潭理素妆”，巧妙运用对比手法，将莲
花不与百花争春的淡泊品格与独自在秋潭中保持本真的高洁气质相映衬，展现了莲花超然物外的精神境界。',
            'joke': 
'莲花最近很不开心，因为大家老拿她和别人比。\n\n有一天，荷叶安慰她：“别郁闷了，你出淤泥而不染，多高尚啊！”\n\n莲花叹
了口气：“高尚有什么用？隔壁那个菊花，人家一夸就是‘采菊东篱下’，多雅致；楼上那个梅花，更是‘凌寒独自开’，多励志。”\n\
n荷叶说：“你也有啊，‘接天莲叶无穷碧，映日荷花别样红’！”\n\n莲花翻了个白眼：“那夸的都是你荷叶好吗？‘接天莲叶’是谁？‘
无穷碧’是谁？我顶多算个‘别样红’，还是个配角！”\n\n荷叶刚想再劝，莲花又说：“最气人的是牡丹，人家直接叫‘花中之王’，我
呢？花中‘廉’洁代表——每次说到我，都跟反腐倡廉绑一块儿，搞得我跟纪委似的。”\n\n这时，池塘边一个小男孩跑过来，对着莲花
大喊：“妈妈快看！那个荷——花——蛋——！”\n\n莲花当场崩溃：“是莲！花！……算了，‘荷花蛋’就‘荷花蛋’吧，至少不是‘反腐蛋’。”'
,
            'final_output': '关于 莲花 
的七言绝句：\n《莲花》\n玉立亭亭水中央，素影涵波暗自香。\n不共春风争颜色，独向秋潭理素妆。\n\n赏析：这首作品以莲花
为吟咏对象，通过“玉立亭亭”、“素影涵波”等意象勾勒出莲花的清雅姿态。后两句“不共春风争颜色，独向秋潭理素妆”，巧妙运用
对比手法，将莲花不与百花争春的淡泊品格与独自在秋潭中保持本真的高洁气质相映衬，展现了莲花超然物外的精神境界。\n笑话
：\n莲花最近很不开心，因为大家老拿她和别人比。\n\n有一天，荷叶安慰她：“别郁闷了，你出淤泥而不染，多高尚啊！”\n\n莲
花叹了口气：“高尚有什么用？隔壁那个菊花，人家一夸就是‘采菊东篱下’，多雅致；楼上那个梅花，更是‘凌寒独自开’，多励志。
”\n\n荷叶说：“你也有啊，‘接天莲叶无穷碧，映日荷花别样红’！”\n\n莲花翻了个白眼：“那夸的都是你荷叶好吗？‘接天莲叶’是
谁？‘无穷碧’是谁？我顶多算个‘别样红’，还是个配角！”\n\n荷叶刚想再劝，莲花又说：“最气人的是牡丹，人家直接叫‘花中之王
’，我呢？花中‘廉’洁代表——每次说到我，都跟反腐倡廉绑一块儿，搞得我跟纪委似的。”\n\n这时，池塘边一个小男孩跑过来，对
着莲花大喊：“妈妈快看！那个荷——花——蛋——！”\n\n莲花当场崩溃：“是莲！花！……算了，‘荷花蛋’就‘荷花蛋’吧，至少不是‘反腐
蛋’。”'
        },
        next=(),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1ad0f4-26e7-6ae8-8002-66df0221fa71'
            }
        },
        metadata={'source': 'loop', 'step': 2, 'parents': {}},
        created_at='2026-09-10T12:00:47.584718+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1ad0f4-26e2-6098-8001-e34e38d800a5'
            }
        },
        tasks=(),
        interrupts=()
    ),
    StateSnapshot(
        values={
            'topic': '莲花',
            'poem': 
'《莲花》\n玉立亭亭水中央，素影涵波暗自香。\n不共春风争颜色，独向秋潭理素妆。\n\n赏析：这首作品以莲花为吟咏对象，通
过“玉立亭亭”、“素影涵波”等意象勾勒出莲花的清雅姿态。后两句“不共春风争颜色，独向秋潭理素妆”，巧妙运用对比手法，将莲
花不与百花争春的淡泊品格与独自在秋潭中保持本真的高洁气质相映衬，展现了莲花超然物外的精神境界。',
            'joke': 
'莲花最近很不开心，因为大家老拿她和别人比。\n\n有一天，荷叶安慰她：“别郁闷了，你出淤泥而不染，多高尚啊！”\n\n莲花叹
了口气：“高尚有什么用？隔壁那个菊花，人家一夸就是‘采菊东篱下’，多雅致；楼上那个梅花，更是‘凌寒独自开’，多励志。”\n\
n荷叶说：“你也有啊，‘接天莲叶无穷碧，映日荷花别样红’！”\n\n莲花翻了个白眼：“那夸的都是你荷叶好吗？‘接天莲叶’是谁？‘
无穷碧’是谁？我顶多算个‘别样红’，还是个配角！”\n\n荷叶刚想再劝，莲花又说：“最气人的是牡丹，人家直接叫‘花中之王’，我
呢？花中‘廉’洁代表——每次说到我，都跟反腐倡廉绑一块儿，搞得我跟纪委似的。”\n\n这时，池塘边一个小男孩跑过来，对着莲花
大喊：“妈妈快看！那个荷——花——蛋——！”\n\n莲花当场崩溃：“是莲！花！……算了，‘荷花蛋’就‘荷花蛋’吧，至少不是‘反腐蛋’。”'
        },
        next=('node_output',),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1ad0f4-26e2-6098-8001-e34e38d800a5'
            }
        },
        metadata={'source': 'loop', 'step': 1, 'parents': {}},
        created_at='2026-09-10T12:00:47.582394+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1ad0f3-fef8-66f4-8000-fb5c3dea8d0b'
            }
        },
        tasks=(
            PregelTask(
                id='54497b55-c07b-83e3-a120-b9acfac1b164',
                name='node_output',
                path=('__pregel_pull', 'node_output'),
                error=None,
                interrupts=(),
                state=None,
                result={
                    'final_output': '关于 莲花 
的七言绝句：\n《莲花》\n玉立亭亭水中央，素影涵波暗自香。\n不共春风争颜色，独向秋潭理素妆。\n\n赏析：这首作品以莲花
为吟咏对象，通过“玉立亭亭”、“素影涵波”等意象勾勒出莲花的清雅姿态。后两句“不共春风争颜色，独向秋潭理素妆”，巧妙运用
对比手法，将莲花不与百花争春的淡泊品格与独自在秋潭中保持本真的高洁气质相映衬，展现了莲花超然物外的精神境界。\n笑话
：\n莲花最近很不开心，因为大家老拿她和别人比。\n\n有一天，荷叶安慰她：“别郁闷了，你出淤泥而不染，多高尚啊！”\n\n莲
花叹了口气：“高尚有什么用？隔壁那个菊花，人家一夸就是‘采菊东篱下’，多雅致；楼上那个梅花，更是‘凌寒独自开’，多励志。
”\n\n荷叶说：“你也有啊，‘接天莲叶无穷碧，映日荷花别样红’！”\n\n莲花翻了个白眼：“那夸的都是你荷叶好吗？‘接天莲叶’是
谁？‘无穷碧’是谁？我顶多算个‘别样红’，还是个配角！”\n\n荷叶刚想再劝，莲花又说：“最气人的是牡丹，人家直接叫‘花中之王
’，我呢？花中‘廉’洁代表——每次说到我，都跟反腐倡廉绑一块

In [7]:
config = {
    "configurable":
        {
            "thread_id": "123",
            'checkpoint_id': '1f1ad0f3-fef8-66f4-8000-fb5c3dea8d0b'
        }
}

get_checkpoints = list(graph.get_state(config=config))
rprint(get_checkpoints)

[
    {'topic': '莲花'},
    ('node_poem', 'node_joke'),
    {'configurable': {'thread_id': '123', 'checkpoint_id': '1f1ad0f3-fef8-66f4-8000-fb5c3dea8d0b'}},
    {'source': 'loop', 'step': 0, 'parents': {}},
    '2026-09-10T12:00:43.397283+00:00',
    {
        'configurable': {
            'thread_id': '123',
            'checkpoint_ns': '',
            'checkpoint_id': '1f1ad0f3-fef6-60a2-bfff-91f3ddd40ad9'
        }
    },
    (
        PregelTask(
            id='7301b3e8-fba2-423f-43f0-b8313c0d12a1',
            name='node_poem',
            path=('__pregel_pull', 'node_poem'),
            error=None,
            interrupts=(),
            state=None,
            result={
                'poem': 
'《莲花》\n玉立亭亭水中央，素影涵波暗自香。\n不共春风争颜色，独向秋潭理素妆。\n\n赏析：这首作品以莲花为吟咏对象，通
过“玉立亭亭”、“素影涵波”等意象勾勒出莲花的清雅姿态。后两句“不共春风争颜色，独向秋潭理素妆”，巧妙运用对比手法，将莲
花不与百花争春的淡泊品格与独自在秋潭中保持本真的高洁气质相映衬，展现了莲花超然物外的精神境界。'
            }
        ),
        PregelTask(
            id='d95956ec-84f3-5a1d-bffe-27d05507ec7e',
            name='node_joke',
            path=('__pregel_pull', 'node_joke'),
            error=None,
            interrupts=(),
            state=None,
            result={
                'joke': 
'莲花最近很不开心，因为大家老拿她和别人比。\n\n有一天，荷叶安慰她：“别郁闷了，你出淤泥而不染，多高尚啊！”\n\n莲花叹
了口气：“高尚有什么用？隔壁那个菊花，人家一夸就是‘采菊东篱下’，多雅致；楼上那个梅花，更是‘凌寒独自开’，多励志。”\n\
n荷叶说：“你也有啊，‘接天莲叶无穷碧，映日荷花别样红’！”\n\n莲花翻了个白眼：“那夸的都是你荷叶好吗？‘接天莲叶’是谁？‘
无穷碧’是谁？我顶多算个‘别样红’，还是个配角！”\n\n荷叶刚想再劝，莲花又说：“最气人的是牡丹，人家直接叫‘花中之王’，我
呢？花中‘廉’洁代表——每次说到我，都跟反腐倡廉绑一块儿，搞得我跟纪委似的。”\n\n这时，池塘边一个小男孩跑过来，对着莲花
大喊：“妈妈快看！那个荷——花——蛋——！”\n\n莲花当场崩溃：“是莲！花！……算了，‘荷花蛋’就‘荷花蛋’吧，至少不是‘反腐蛋’。”'
            }
        )
    ),
    ()
]